<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures/3_Lecture_NN_tabular_data_DeeperDive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nuts and Bolts - Neural Networks for Structured Data or Tabular data
Feed Forward Neural Network (FFNN) / Multi-layer Perceptrons (MLP)
References:
* Chapter 13, Probabilistic Machine Learning: An Introduction by Kevin Murphy  
* Bengio, Y., Practical recommendations for gradient-based training of deep architectures, 2012, https://arxiv.org/abs/1206.5533
* Nawankpa, C., Activation Functions: Comparison of trends in Practice and Research for Deep Learning, 2017, https://doi.org/10.48550/arXiv.1811.03378

---


# **Recommended readings**
* Bengio, Y., Practical recommendations for gradient-based training of deep architectures, 2012, https://arxiv.org/abs/1206.5533
* Nawankpa, C., Activation Functions: Comparison of trends in Practice and Research for Deep Learning, 2017, https://doi.org/10.48550/arXiv.1811.03378

---

# **Additional resources**
* [Visualizing piecewise linear neural networks](https://blog.janestreet.com/visualizing-piecewise-linear-neural-networks/?utm_source=conference&utm_medium=booth&utm_campaign=ICML)
* Optimizers: [Computational implementation in Pytorch](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html#torch.optim.SGD)  ||
[Visualization tool](https://github.com/lilipads/gradient_descent_viz)

---
---

### **Outline**
* Data handling
* Data preprocessing
* Split data into test and train  
* Batching of train set
* Activation functions
* Regularization (avoid overfitting)
* Optimizers
* Hyperparameter tuning
* Loss functions


---

# **I. Data preprocessing**
 * Data standardization, or Z-score normalization, scales data to have a mean of 0 and a standard deviation of 1, preserving the distribution shape but without a fixed range.
 * Data normalization, such as Min-Max Scaling, scales data to a specific range, typically 0 to 1, which can alter the distribution's shape and is useful for algorithms that require data within a certain range

 In deep learning we prefer standardization aas it preserves the distribution

 ---

# **II. Train and test sets**
* Typical to split data into train and test sets
* Use ‘train’ set to train the data
* Test the trained model on the ‘test’ set
---

# **III. Batching of train set**

* Batch: use the full train set to train the model
* Mini-batch: divide train set into small mini-batches
  * Typically $2^n$: 32, 64, 128, 256 (corresponding to CPU /GPU architecture)
* Incremental/ online learning: single sample at a time

Bengio, Practical recommendations for gradient-based training of deep architectures, 2012, https://arxiv.org/abs/1206.5533

---

# **IV. Activation functions**
Main properties of suitable activation functions:
* Non-linear functions
* Continuously Differentiable

Common used functions
\begin{array}{|l|l|l|l|}
\hline
\textbf{Name} & \textbf{Definition} & \textbf{Range} & \textbf{Reference} \\
\hline
\text{Sigmoid} & \sigma(a) = \frac{1}{1 + e^{-a}} & [0, 1] &  \\
\hline
\text{Hyperbolic tangent} & \tanh(a) = 2\sigma(2a) - 1 & [-1, 1] &  \\
\hline
\text{Softplus} & \sigma_{+}(a) = \log(1 + e^{a}) & [0, \infty) & \text{[GBB11]} \\
\hline
\text{Rectified linear unit} & \mathrm{ReLU}(a) = \max(a, 0) & [0, \infty) & \text{[GBB11; KSH12]} \\
\hline
\text{Leaky ReLU} & \max(a, 0) + \alpha \min(a, 0) & (-\infty, \infty) & \text{[MHN13]} \\
\hline
\text{Exponential linear unit} & \max(a, 0) + \min\big(\alpha(e^{a} - 1), 0\big) & (-\infty, \infty) & \text{[CUH16]} \\
\hline
\text{Swish} & a \, \sigma(a) & (-\infty, \infty) & \text{[RZL17]} \\
\hline
\text{GELU} & a \, \Phi(a) & (-\infty, \infty) & \text{[HG16]} \\
\hline
\end{array}
*Table Source:* Reproduced from Table 13.4 PML: An Introduction by Murphy

<p align="center">
  <img src="https://raw.githubusercontent.com/probml/pml-book/main/book1-figures/Figure_13.14_A.png" width="45%" />
  <img src="https://raw.githubusercontent.com/probml/pml-book/main/book1-figures/Figure_13.14_B.png" width="45%" />
</p>

<p align="center"><em>Figure 13.14 from PML: An Introduction by Murphy (Directly embedded figure from the textbook's GitHub repository)</em></p>


---

## Types of Activation functions

## Common Saturating functions: Sigmoid and Tanh
* Were the most commonly used functions in the early days, as
  * They were continuously differentiable 'i.e., differentiable at every point', and
  * The smooth representation between 0 and 1 were suitable to represent most functions.
  * Vanishing and Exploding Gradient issues were not a problem previously as neural nets were pretty shallow (one or two hidden layers) (limited by compute capabilities).
* They are both saturating functions
  * Sigmoid saturates at 1 for large positive inputs, and at 0 for large negative inputs.
  * Tanh function, has a similar shape at Sigmoid, but saturates at -1 and +1.

###  **Vanishing/Exploding Gradient Issues in Saturating functions**

When using saturated activation functions, the gradient of the output wrt the input may become either very small (this is called the vanishing gradient problem) or very large (this is called the exploding gradient problem), as gradients are propogated backward through many layers. Thus, the gradient signal may vanish (not propagate back) or explode (produce infinite values of weights).

Solution:
* Modern day solution: use **non-saturating** functions
* Also doable for simple cases: INitialize weights to be not too high or too low, and use shallow layers if feasible.
* The exploding gradient problem can be fixed by **gradient clipping** $g' = min(1,\frac{c}{||g||})g $  
  * $g$ is the gradent at some layer; the equation ensures that the norm of $ g$ is never greater than some constant $c$; and multiplying by $g$ ensures the vector is in same direction as $g$
* Use non-saturating activation functions


---



## Common Non-saturating Activation functions
### **1. ReLU (Rectified Linear Unit)**

  The ReLU function simply “turns off” negative inputs, and passes positive inputs unchanged.
  $ReLU(a) = max(a, 0) =a\mathbb{I}(a>0); \mathbb{I}$ is an indicator function.  
  $ReLU'(a) = \mathbb{I}(a>0)$  
  Suppose $f = ReLU(a)= ReLU(Wx)$   
  $\frac{\partial f }{\partial W}=\mathbb{I}(a>0)=\mathbb{I}(Wx>0)x^T $

Another way of writing the same
  $$
f(a; \alpha) =\mathrm{RELU}(a; \alpha) =
\begin{cases}
0, & \text{if } a \leq 0 \\
a, & \text{if } a > 0
\end{cases}
$$
  $$
\frac{df(a; \alpha)}{da} =
\begin{cases}
0, & \text{if } a \leq 0 \\
1, & \text{if } a > 0
\end{cases}
$$

  Suppose $f = ReLU(a)= ReLU(W_1x_1+W_2x_2+W_3x_3)$   
$$
\frac{df}{dW_1} =
\begin{cases}
0, & \text{if } a \leq 0 \\
x_1, & \text{if } a > 0
\end{cases}
$$

$$
\frac{df}{dW_2} =
\begin{cases}
0, & \text{if } a \leq 0 \\
x_2, & \text{if } a > 0
\end{cases}
$$

$$
\frac{df}{dW_2} =
\begin{cases}
0, & \text{if } a \leq 0 \\
x_3, & \text{if } a > 0
\end{cases}
$$

IN VECTOR FORM: Suppose $a=Wx$;   
$x$ is a column vector, and $W$ is a row vector (the linear transformation in any NN layer)
  $$
\frac{df}{dW} =
\begin{cases}
0, & \text{if } a \leq 0 \\
x^T, & \text{if } a > 0
\end{cases}
$$


### **. Dead ReLU problem**

When using ReLU, if the weights are initialized such that $a = Wx$ take on large negative values, then the signal outputs from neurons are never activated, and the signal dies out . This is called the **dead ReLU** problem

### **2. Leaky ReLU : Non-saturating version of ReLU**  
Overcomes dead ReLU problem.
$LReLU(a; \alpha) = max(\alpha a, a)$  
where $0 < \alpha < 1$. The slope of this function is 1 for positive inputs, and $\alpha$ for negative inputs, thus ensuring there is some signal passed back to earlier layers, even when the input is negative.

$\alpha$ is a **hyperparameter**

### **3. Exponential Linear Unit (ELU)**
$$
f(a; \alpha) =\mathrm{ELU}(a; \alpha) =
\begin{cases}
\alpha \big(e^{a} - 1\big), & \text{if } a \leq 0 \\
a, & \text{if } a > 0
\end{cases}
$$

$$
\frac{df(a; \alpha)}{da} =
\begin{cases}
\alpha e^{a} , & \text{if } a \leq 0 \\
1, & \text{if } a > 0
\end{cases}
$$
Allow for smooth derivatives.

Suppose $a=Wx$;   
$x$ is a column vector, and $W$ is a row vector (the linear transformation in any NN layer)
$$
f(Wx; \alpha) =\mathrm{ELU}(Wx; \alpha) =
\begin{cases}
\alpha \big(e^{Wx} - 1\big), & \text{if } Wx \leq 0 \\
a, & \text{if } Wx > 0
\end{cases}
$$

$$
\frac{\partial f(Wx; \alpha)}{\partial W} =
\begin{cases}
\alpha e^{Wx} x, & \text{if } a \leq 0 \\
 x, & \text{if } a > 0
\end{cases}
$$


### 4. **SELU (Self-normalizing ELU**  )
$SELU(a; α, λ) = λELU(a; α)$  
The authors prove that by setting $\alpha$ and $\lambda$ to carefully chosen values, this activation function is guaranteed to ensure that the output of each layer is standardized (provided the input is also standardized), even without the use of techniques such as batchnorm

---

QUESTION: are the non-saturating functions, continuously differentiable? (a requirement for NN?). If not why are they acceptable?

# **V. Optimizers (See SLIDES )**
Different options for search direction and learning rate (step size)
### Line search methods
Methods that pick a search direction and step in that direction with some step size. In ML step-size are typically referred to as learing rate
* Gradient as search direction
  * SDG
  * SDG with momentum
  * SDG with Nestrov
* Adaptive learning rate (step size)
  * AdaGrad
  * RMSProp
  * Adam
* Hessian as search direction
  * Newtons method
  * Conjugate gradients
  * BFGS

### Trust-region methods
Methods that determine search direction and step-size together
* Levenberg Marquardt

**See SLIDES on Canvas**

### References
[Algorithms](https://www.deeplearningbook.org/contents/optimization.html) Algorithms 8.1 to 8.7 from Chapter 8: Ian Goodfellow and Yoshua Bengio and Aaron Courville, Deep Learning  
[Computational implementation in Pytorch](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html#torch.optim.SGD)  

[Vizualization tool](https://github.com/lilipads/gradient_descent_viz)


(See convergence properties for optimizers [1suppl_Mathematical Foundations of ML](https://github.com/chaitragopalappa/MIE590-690D/blob/main/suppl_files/1suppl_Mathematical_foundations_of_ML.ipynb)

---
### **Optimizers -Additional variants**

New variants are continuoualy added- Best way to keep up is to look at NN libraries (Keras, Pytorch, Tensorflow) on available options
* [Kieras](https://keras.io/api/optimizers/)  
* [Pytorch](https://pytorch.org/docs/stable/optim.html)  
* [Tensorflow](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers )  

Bengio, Practical recommendations for gradient-based training of deep architectures, 2012, https://arxiv.org/abs/1206.5533

---

# **VI. Architecture for regression v. classification**

* Regression is the machine learning term used to indicate that the response variable takes Real values, i.e., $y\in \mathbb{R}$, or is a continuous variable
* Classification problem is when the response variable is a discrete variable. It can be binary (0/1), categorical-nominal (e.g., Race), or categorical-ordinal (e.g., bad, good, excellent)


<p align="center">
  <img src="https://raw.github.com/diseasemodeling/MIE_525_625/main/figs/Reg_Class.png"  />

</p>



## **Architecture: MLP on tabular data : regression problem**

The NN architecture we have reviewed in slideset 2 assumed it is a regression problem.
We had the following (matrix form):

Suppose
 $y=f(\mathbf{x})$  
 Predicted $\hat{y}$:  
 $ \hat{y}= \mathbf{z}_L=(\mathbf{b}_{L} +\mathbf{W}_{L}\mathbf{z}_{L−1})$  
 $\mathbf{z}_l=  \phi_l (\mathbf{b}_{l} +\mathbf{W}_{l}\mathbf{z}_{l−1})$  
 $l=1:L-1$ hidden layers;   
 $z_0=x$

## **Architecture: MLP for tabular data : classification problem**

**TWo classes**  
For two-class problem, we can still use only one node in the output, and assume it estimates the logit function, and pass it through sigmoid function to get the probability.
 $$logit(p)=ln(\frac{p}{1−p})=x$$
 $$\text{sigmoid function}~\sigma(x)=\frac{1}{1+e^{−x}}=p$$
 IN statistics, a logit is the natural logarithm of the odds, where odds are the ratio of the probability of an event occurring to the probability of it not occurring. The logit function transforms a probability (a value between 0 and 1) into a value that can range from negative to positive infinity. IN deep learning, we do the inverse, pass the logit either through sigmoid (for 2-class problem) or softmax (for N-class problem) to convert outputs to a probability dsitribution. Thus, the output from the NN are equivalent to learning the logits.



Thus, the output from the final layer in a classification problem (before passing through a softmax or logistic function) are called the **logit scores**  

 **Multi-class classification (> 2 classes)**  
Same as above except logit scores are passed through a soft-max function to generate a probability distribution that adds to 1 (sigmoid only converts it into values between 0 and 1, so only used in 2-class problem. In 2-class problem, if we know probability of class 1 (p) we can calculate probability of class 2 as  1-p)

In NN packages: soft-max is typically integrated into the cross-entropy loss, and thus, in setting-up NN archtecture,  we need not dpass the final layer through soft-max.


 ---

# **VII. Loss function for MLP regression v. classification problems**


## **VII.a.: Loss function for regression is MSE**

Objective function: $ Min\mathcal{L}(\mathbf{\theta}) =Min_\mathbf{\theta}||\mathbf{\hat{y}-y}||_2^2$

* As discussed in previous chapter, in deep learning, the objective is to find the values of neural network weights ($\theta$), that minimize the Loss function ($\mathcal{L}$), and thus LHS is written as  $ Min\mathcal{L}(\mathbf{\theta})$.

* Notice that, in the RHS, $||.||_2$ is the notation for L2 norm, typically defined for vector $\mathbf{x}$ as $||\mathbf{x}||_2=\sqrt(\sum_{i=1}^N x_i)$. And thus,  $||\mathbf{\hat{y}-y}||_2^2$ is the MSE (mean square error)

## **VII.b. Loss function for classification: cross-entropy loss or log-loss**
We typically encode categorical variables as 0,1,2, etc, as model needs 'math /numbers' not text.
  * Example 1 : 0=White, 1=Black, 2=Alaskan native, etc.
  * Example 2:  0=bad, 1=good, 2= excellent

But training the model as if the response variable takes integer values (i.e., one node in the output layer) will be incorrect, because
* in math, $2>1>0,$ and $2-1=1-0$, but in above examples that does not apply.
* if we use MSE loss, we have $2-0 > 1-0$.
  * how to tell the NN that $2-1$ is not greater than $1-0$? (i.e., the Loss of predicting 2 when actual value is 0, shouldn't be higher than the Loss of predicting 1)? we do this by making the output from NN a probability, i.e., $Pr(y=c)$ (see figure above)

So we instead estimate the 'probability' it belongs to a 'class', i.e., the probability distribution over class.

But the response variable it is trained on is encoded as an integer,
* how to tell the NN to output a probability distribution (each class value should be between 0 and 1, and sum over all classes should add to 1)?
* how to determine what the loss is? (we need the differnce between two probability distributions)

We pass NN outputs through a softmax function to make it a probability distribution. We use a cross-entropy loss to compare distributions.

### Softmax function
To ensure $p_{i,c}$ is actually a probability, and $\sum_cp_{i,c}=1$, at the output node we apply a softmax function (so this is not really an activation function as the main role of an AF is to add non-linearity;).
  
$$Pr(Y=c) =Softmax(\mathbf{y_c}) = \frac{e^{y_c}}{\sum_{j=1}^{C} e^{y_j}}$$

### **What is entropy (Shannon entropy)?**
It indicates how much information we can gather about a random variable from its distribution. A uniform distrubtion has highest entroy, meaning highest uncertainty in what we expect the output to be.
$$H(Y)=- \sum_{c}y_{c}log(y_{c})$$  

### **What is cross-entropy, and relative-entropy (or Kullback-Leibler (KL) Divergence)?**  
Suppose we have two random variables, $Y, P$. KL divergence (also called relative entropy) measures how different the probability distribution of $P$ is from the probability distribution of $Y$.


KL divergence = cross-entropy $-$ entropy = $$H(Y,P) - H(Y) = -\sum_{c}y_{c}log(p_{c}) - (-\sum_{c}y_{c}log(y_{c}))$$  
where, $$y_c=Pr(Y=c); p_c=Pr(P=c)$$

This could be a good Loss function. But why use cross-entropy instead of KL-divergence?

### **Why use Cross-Entropy loss (also called log-loss) for classification?**
Suppose,
  $p_{i,c}$ is the model predicted probability that sample $i$ is in class $c$;  
  $y_{i,c}$ is the actual probability that sample $i$ is in class $c$  
   $
    y_{i,c} =
    \begin{cases}
    1, & \text{if true label is c} \\
    0, & \text{otherwise }
    \end{cases}     
  $  

Loss should be the differnce in the distributions (KL divergence).
But here, minimizing cross-entropy (i.e., $-H(Y,P)$ is equivalent to minimizing KL divergence because $H(Y)=0$  
Thus, $-H(Y,P) = - (H(Y,P) - H(Y))$

Thus, in a classification problem,

   $$ \mathcal{L}=-\frac{1}{N}\sum_{i=1}^N\sum_c y_{i,c} log(p_{i,c})$$






  
---

**## Handling imbalanced data in classification problems**

What is imbalanced data: when some classes are much more common than others, e.g., 90% instances of class A and 10% instances of class B. This is problem because even a simple classifier of assigning A will lead to 90% accuracy. Thus, if we do not observe that there is a class imbalance and our ML model predcts close to 90% - we may be in a false sense of acuracy.

Training or modeling strategies to address class imbalance
- Resampling: oversample minority class (e.g., SMOTE) or undersample majority class to balance training set — helps model see more minority examples but may overfit or lose data.
-	Class weighting / cost-sensitive loss: assign higher loss weight to minority class so model penalizes misclassifying it more — useful without changing data.
-	Algorithmic / loss modifications (e.g., focal loss): focus learning on hard examples by down-weighting easy negatives — helps when many easy negatives dominate gradient.

Instead of macro-average F1 use micro-average F1 score
- Macro-F1: compute F1 per class then average — treats all classes equally (good when per-class performance equally important).
- Micro-F1: aggregate counts across classes then compute F1 — weights by support (good when overall instance-level performance is of interest and classes are imbalanced).

---
Question: why not directly use F1 score as the loss function?

---
---




# **VIII. Regularization- steps to avoid over-fitting**
Regularization refers to methods that can be used to avoid overfitting the model to data.
1. **Early stop**: stopping the training procedure when the error on the validation set starts to increase
2. **Weight decay**:  This is equivalent to L2 regularization (in ridge regression)-  by adding a penalty to the loss function based on the sum of the squares of the model's weights. In the neural network literature, this is called weight decay, since it encourages small weights, and hence simpler models. (This is equivalent to using a Gaussian prior for the weights $\mathcal{N} (w|0, α^2I)$ and biases,$\mathcal{N} (b|0, β^2I)$.)
$$L_m = L +\lambda \sum_w ||w||^2_2 $$
$L_m$ is modified Loss
  *  IN NN packages this is an input to the Optimizer
3. **Drop-out**: Turn off all the outgoing connections from each neuron with probability $p$. Can dramatically reduces over-fitting and us widely used. Intuitively, each unit must learn to perform well even if some of the other units are missing at random.


___


# *IX. Hyperparameter tuning for convergence**

* **What is hyperparameter tuning?**: It involves tuning all the user-input values of the neural network training to improve convergence
* NUMERICAL OPTIMIZATION DOMAIN
  * **What is convergence?** It a **numerical search optimization** terminology to indicate the algorithms has found the optimal solution. The term 'converged' arises from the fact than the derivation of the function approachs zero and flattens at zero. In machine learning the loss function flattens when it stops learning.
  * **What is thedifference between global optmia and local optima?** 'Global optima is the true solution. In numerical optimization, one can never gurantee global optima
  * **Does convergence mean it found the 'global' optima?** No, it refers to convergence to local optima, as we can never guarantee global optima
  * **Then how do we know if our solution is good enough**? We dont, we can attempt to find a better solution by trying differnt starting points and differnt learning rates
* MACHINE LEARNING DOMAIN
  * Optimizers are algorithms from machine learning. Convergence is carried-over
  * **When we see a loss function flatten out does it mean it has convereged?**  NO.
  * **Then how do we know if our solution is good enough**? We dont, this is why hyperparameter tuning is done to attempt to find a better solution.

WHat paramters are tunable in Deep Learning
* Tune parameters related to optimizer, weight decay (regularization weights)
* Use different optimizers
* Use different random seeds if fixing the seed (this will initiate all place where random numbers are generated, e.g., initializing weights)
* Modify neural network architecture
  * number of hidden layers, and nodesUse
  * different activation functions if you suspect issues there



---


# X. MLP for heteroskedastic regression
“Heteroskedastic”  means that the predicted output variance is input-dependent. This function has two outputs which compute $f_{\mu}(x) = \mathbb{E} [y|x, θ]$ and $f_{\sigma}(x) = \sqrt{\mathbb{V} [y|x, θ]}$.
Most of the layers (and hence parameters) can be shared between the two functions by using a common “backbone” and two output “heads”. A linear activation for the $\mu$ head, a softplus activation for the $\sigma$ head are typically used.
![](https://brendanhasz.github.io/assets/img/dual-headed/TwoHeadedNet.svg)
Source: Brandan Hasz, Trip Duration Prediction using Bayesian Neural Networks and TensorFlow 2.0, https://brendanhasz.github.io/2019/07/23/bayesian-density-net.html
